In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
sns.set(style='whitegrid',font_scale=2)
import os
import pickle
import time

os.getcwd()
os.chdir(os.getcwd())
os.getcwd()

from itables import init_notebook_mode
init_notebook_mode(connected=True,all_interactive=True)

----------------------------------------------------------------------------
#### 1. Load subsetted dataframes and create nested dictionaries for the different types of data in form of:
<br>    Datasets with temporal measurement data<br>
<br>       - {pat_id: 	‘bulk_data’: bulk_df_of_patient_data,<br>
<br>                     ‘variables’: list or dict of variables,<br>
<br>                  ‘grouped_data’: DataFrameGroupBy or dict of DataFrameGroupBy}<br>

###   Demographic data with 'constant' data (i.e. Sex, Race, etc.)
<br>     - {pat_id:  'bulk_data':bulk_df_of_patient_data,<br>
<br>                 ‘variables’: list or dict of variables,<br>
<br>                'grouped_data’: df containing only the values of variables<br>

<br>  - BULK_DATA: some dataframes contain multiple types of information (i.e.: pulse, blood pressure, weight,etc.) in one column. Later on, if needed these dataframes will be split up by the type of data <br>
<br> - VARIABLES: measured quality of interest, depends on the type of dataset (i.e. Pulse, Drug treatment, Medical condition, etc)<br>
<br> - GROUPED_DATA: DataFrameGroupBy object(s) which are grouped by the variables<br>

#### 2. Save the dictionaries as "data/ttp_typeofdf_dict"

In [2]:
# load the subsetted datasets

mb=pd.read_csv('../data/out_mb_wo_false_positives.csv.gz',low_memory=False)
#pc=pd.read_csv('../data/out_pc_preprocessed.csv',low_memory=False)
#pc_peaks=pd.read_csv('../data/ttp_pc_peaks.csv',low_memory=False)
#pc_baseline=pd.read_csv('../data/ttp_pc_baseline.csv',low_memory=False)
dm=pd.read_csv('../data/out_dm.csv.gz',low_memory=False)
vs=pd.read_csv('../data/out_vs_standardised.csv.gz',low_memory=False)
re=pd.read_csv('../data/out_re_standardised.csv.gz',low_memory=False)
lb=pd.read_csv('../data/out_lb.csv.gz',low_memory=False)
mh=pd.read_csv('../data/out_mh_standardised.csv.gz',low_memory=False)
ce=pd.read_csv('../data/out_ce_standardised_with_time.csv.gz',low_memory=False)
cm=pd.read_csv('../data/out_cm_standardised_with_drugs.csv.gz',low_memory=False) # use this dataset for the concomitant medication data
cmind=cm.copy()                          # use this dataset for the indications for the concomitant medication
ms=pd.read_csv('../data/out_ms.csv.gz',low_memory=False)
ae=pd.read_csv('../data/out_ae_standardised.csv.gz',low_memory=False)
su=pd.read_csv('../data/out_su.csv.gz',low_memory=False)
#de=pd.read_csv(dir+'//ttp_de.csv',low_memory=False,index_col=0)

#load patient ids
pat_ids=pd.read_csv('../data/patients_in_analysis.csv.gz',index_col=0)
#pat_ids=pat_ids['USUBJID'].values.tolist()

#get drug names used in studies
#f = open(dir+'/ttp_all_drugs.pkl','rb')
#ttp_all_drugs=pickle.load(f)
#f.close()


#split up microbiolgical susceptilibilty dataset
mr=ms[ms['STD_MSTEST']=='Molecular Drug Resistance']
mic=ms[ms['STD_MSTEST']=='Minimum Inhibitory Concentration']
ms=ms[ms['STD_MSTEST']=='Microbial Susceptibility']



In [3]:
a=mb[mb[['USUBJID','MBDY_estimated','MBTSTDTL']].duplicated(keep=False)].sort_values(['USUBJID','MBDY_estimated']).dropna(how='all',axis=0)
a #groupby('MBTSTDTL').apply(lambda x: x)

Unnamed: 0  STUDYID DOMAIN       USUBJID SPDEVID  MBSEQ MBGRPID  \
426004      426037  TB-1001     MB  TB-1001/0018     NaN      1     NaN   
427155      427188  TB-1001     MB  TB-1001/0018     NaN      2     NaN   
426007      426040  TB-1001     MB  TB-1001/0029     NaN      1     NaN   
427158      427191  TB-1001     MB  TB-1001/0029     NaN      2     NaN   
426008      426041  TB-1001     MB  TB-1001/0031     NaN      1     NaN   
...            ...      ...    ...           ...     ...    ...     ...   
328936      328936  TB-1024     MB      TX099688     NaN     31     NaN   
329453      329453  TB-1024     MB      TX099688     NaN     32     NaN   
329805      329805  TB-1024     MB      TX099688     NaN     33     NaN   
330147      330147  TB-1024     MB      TX099688     NaN     34     NaN   
330497      330497  TB-1024     MB      TX099688     NaN     35     NaN   

       MBREFID MBSPID MBLNKID  ... COMMENT SUSCDONE    RESULT STD_NUM_RESULT  \
426004     NaN    NaN     NaN  ...     NaN      NaN  positive            NaN   
427155     NaN    NaN     NaN  ...     NaN      NaN  positive            NaN   
426007     NaN    NaN     NaN  ...     NaN      NaN  positive            NaN   
427158     NaN    NaN     NaN  ...     NaN      NaN  positive            NaN   
426008     NaN    NaN     NaN  ...     NaN      NaN  positive            NaN   
...        ...    ...     ...  ...     ...      ...       ...            ...   
328936   26193    NaN     NaN  ...     NaN      NaN  negative            0.0   
329453   29794    NaN     NaN  ...     NaN      NaN  negative            NaN   
329805   29795    NaN     NaN  ...     NaN      NaN  negative            NaN   
330147   29794    NaN     NaN  ...     NaN      NaN  negative            0.0   
330497   29795    NaN     NaN  ...     NaN      NaN  negative            0.0   

       STD_NUM_UNITS STD_CAT_RESULT STD_CAT_UNITS  MBDY_estimated  \
426004           NaN             3+           NaN           -78.0   
427155           NaN             4+           NaN           -78.0   
426007           NaN             2+           NaN           -62.0   
427158           NaN             3+           NaN           -62.0   
426008           NaN             4+           NaN           -63.0   
...              ...            ...           ...             ...   
328936           CFU              0           CFU           169.0   
329453           NaN       NEGATIVE           NaN           333.0   
329805           NaN       NEGATIVE           NaN           333.0   
330147           CFU              0           CFU           333.0   
330497           CFU              0           CFU           333.0   

                         ARM   SAMPLE_REFID  
426004  RIFAPENTINE + INH QS            NaN  
427155                   NaN            NaN  
426007    RIFAMPIN + INH BIS            NaN  
427158                   NaN            NaN  
426008  RIFAPENTINE + INH QS            NaN  
...                      ...            ...  
328936                   NaN  TB-1024_26193  
329453                   NaN  TB-1024_29794  
329805                   NaN  TB-1024_29795  
330147                   NaN  TB-1024_29794  
330497                   NaN  TB-1024_29795  

[268767 rows x 54 columns]

##### 1. Datasets have to be grouped by variables-> this dictionary contains the names of the columns that contain the variables.
##### 2. Create a dataframe containing the variables for each dataset

In [3]:
ttp_datasets=[mb,dm,vs,re,lb,mh,ce,cm,cmind,mr,mic,ms,ae,su]
ttp_datasets_name=['mb','dm','vs','re','lb','mh','ce','cm','cmind','mr','mic','ms','ae','su']
groupby_colnames={ 'mb':'STD_MBTEST',
                   'dm':['STUDYID','AGE','SEX','RACE','ARM'],
                   #'pc':'PCTEST',
                   'vs':'STD_VSTEST',
                   're':'STD_RETEST',
                   'lb':'LBTEST',
                   'mh':'ALL_STD_TERMS',
                   'ce':'STD_CETERM',
                   'cm':'STD_DRUGS_REPLACED',
                   'cmind':'STD_CMINDC',
                   'mr':'STD_MSAGENT',
                   'mic':'STD_MSAGENT',
                   'ms':'STD_MSAGENT',
                   'ae':'STD_AETERM',
                   'su':'STD_SUTRT'}

## 

## CHOOSE THE VARIABLES THAT HAVE VALUES IN MORE THAN 'N' PATIENTS PER EACH PHASE

In [4]:
#load all the variables per patient dataframe
variables_per_patient_all=pd.read_csv('../data/all_pat_variables.csv.gz',index_col=0,low_memory=False)

#load patient IDs who are considered in this  analysis
pat_id_df=pd.read_csv('../data/patients_in_analysis.csv.gz',index_col=0)

# get all pat ids
all_ids=pat_id_df['USUBJID'].to_list()

#check how many variables are available for the considered patients
variables_per_patient_ttp=variables_per_patient_all.loc[pat_id_df['USUBJID'],]
var_nums=variables_per_patient_ttp.sum().sort_values(ascending=False).to_frame()

## Create dictionary to collect variables that are used for the analysis for each phase
vars_for_analysis_per_phase={}


# Check how many variables are available for the considered patients
for phase in [3]:
    df_phase=pat_id_df[(pat_id_df['STUDYID']!='TB-1031')]
    
    variables_per_patient_ttp=variables_per_patient_all.loc[df_phase['USUBJID'],:]
    var_nums=variables_per_patient_ttp.sum().sort_values(ascending=False).to_frame()

    ### CHOOSE THE VARIABLES THAT HAVE VALUES IN MORE THAN 'N' PATIENTS PER EACH PHASE
    
    N=0
    
    ####
    
    var_nums=var_nums[var_nums[0]>N] #0.1*max(var_nums[0]) 
    ## Create data about the number of variables for available for how many patients
    var_nums_for_patients=var_nums.value_counts().sort_index(ascending=False)
    data=var_nums_for_patients.cumsum()

    '''
    fig,ax=plt.subplots(1,1,figsize=(12,len(data)*0.5))
    sns.barplot(x=data.index.get_level_values(0).tolist(),y=data.values,ax=ax,orient='h')
    ax.set_ylabel('Number of variables')
    ax.set_xlabel('Number of patients with variables')
    ax.set_title('Variables available for patients in phase '+ str(phase))
    ax.bar_label(ax.containers[0])
    '''
    ## Add variables that are considered for each phase to a dictionary -> use this in the next step to create dictionary containing 
    #  the available variable names and variable values for each patient
    vars_for_analysis_per_phase['phase_'+str(phase)]=var_nums.sort_index().index.to_list() 

    # Save vars_for_analysis_per_phase
    with open('../data/vars_for_analysis_per_phase.pkl', 'wb') as file:
        pickle.dump(vars_for_analysis_per_phase, file)

AGE  ARM  RACE  SEX  STUDYID  \
TB-1020/1016     1.0    1   1.0    1        1   
TB-1020/1023     1.0    1   1.0    1        1   
TB-1020/1029     1.0    1   1.0    1        1   
TB-1020/1031     1.0    1   1.0    1        1   
TB-1020/1039     1.0    1   1.0    1        1   
...              ...  ...   ...  ...      ...   
TB-1018/04-9011  1.0    1   1.0    1        1   
TB-1018/04-9010  1.0    1   1.0    1        1   
TB-1018/02-9066  1.0    1   1.0    1        1   
TB-1018/02-9046  1.0    1   1.0    1        1   
TB-1018/02-9053  1.0    1   1.0    1        1   

                 ae_( R ) UPPER LOBE CREPITATIONS  \
TB-1020/1016                                  NaN   
TB-1020/1023                                  NaN   
TB-1020/1029                                  NaN   
TB-1020/1031                                  NaN   
TB-1020/1039                                  NaN   
...                                           ...   
TB-1018/04-9011                               NaN   
TB-1018/04-9010                               NaN   
TB-1018/02-9066                               NaN   
TB-1018/02-9046                               NaN   
TB-1018/02-9053                               NaN   

                 ae_(CONJUCTIVITIS) BILATERAL  ae_(FEET) FUNGAL INFECTION  \
TB-1020/1016                              NaN                         NaN   
TB-1020/1023                              NaN                         NaN   
TB-1020/1029                              NaN                         NaN   
TB-1020/1031                              NaN                         NaN   
TB-1020/1039                              NaN                         NaN   
...                                       ...                         ...   
TB-1018/04-9011                           NaN                         NaN   
TB-1018/04-9010                           NaN                         NaN   
TB-1018/02-9066                           NaN                         NaN   
TB-1018/02-9046                           NaN                         NaN   
TB-1018/02-9053                           NaN                         NaN   

                 ae_(IMPETIGO)RASH ON LOWER LIMBS  \
TB-1020/1016                                  NaN   
TB-1020/1023                                  NaN   
TB-1020/1029                                  NaN   
TB-1020/1031                                  NaN   
TB-1020/1039                                  NaN   
...                                           ...   
TB-1018/04-9011                               NaN   
TB-1018/04-9010                               NaN   
TB-1018/02-9066                               NaN   
TB-1018/02-9046                               NaN   
TB-1018/02-9053                               NaN   

                 ae_(L)CLAVICLE INJURY DUE TO BLUNT TRAUMA  ...  \
TB-1020/1016                                           NaN  ...   
TB-1020/1023                                           NaN  ...   
TB-1020/1029                                           NaN  ...   
TB-1020/1031                                           NaN  ...   
TB-1020/1039                                           NaN  ...   
...                                                    ...  ...   
TB-1018/04-9011                                        NaN  ...   
TB-1018/04-9010                                        NaN  ...   
TB-1018/02-9066                                        NaN  ...   
TB-1018/02-9046                                        NaN  ...   
TB-1018/02-9053                                        NaN  ...   

                 re_Zone Score  su_SMOKING EVER  vs_Body Mass Index  \
TB-1020/1016               NaN              1.0                 NaN   
TB-1020/1023               NaN              1.0                 NaN   
TB-1020/1029               NaN              1.0                 NaN   
TB-1020/1031               NaN              1.0                 NaN   
TB-1020/1039               NaN              1.0                 NaN  

In [12]:
variables_per_patient_all_ = variables_per_patient_all.replace(0,np.nan)
variables_per_patient_all_.loc['TB-1020/1016',:].dropna()


AGE                                    1.0
ARM                                    1.0
RACE                                   1.0
SEX                                    1.0
STUDYID                                1.0
lb_Blood Alanine Aminotransferase      1.0
lb_Blood Aspartate Aminotransferase    1.0
lb_Blood Creatinine                    1.0
lb_Blood Creatinine Clearance          1.0
lb_Blood Hemoglobin                    1.0
mb_MGIT                                1.0
mb_ZN-smear                            1.0
re_Cavitation                          1.0
su_SMOKING EVER                        1.0
vs_Weight                              1.0
Name: TB-1020/1016, dtype: float64

## __Create a dictionary, which contains the name of variables + variable values available for each patient__
- #### __dict[pat_id]['variables']__= list of variables available for the patient
- #### __dict[pat_id]['grouped_data']__= dataframes grouped by the variables (made with pandas.groupby function)

In [5]:
import time
start = time.time()
import warnings
warnings.filterwarnings('ignore')
    
for ds,ds_name in zip(ttp_datasets[0:],[*groupby_colnames][0:]):
#for ds,ds_name in zip([mb],['mb']):    
    print(ds_name)
    ds_dict={}
    for phase in [3]:
        vars_for_analysis_in_phase=vars_for_analysis_per_phase['phase_'+str(phase)]
        pats_in_phase=pat_ids.loc[:,'USUBJID'].values.tolist()
        common_ids=list(set(pats_in_phase)&set(ds['USUBJID'].values))
        
        # Demographic dataset doesn't have temporal measurement points, only 'constants' -> 
        # look up in the description for data structure here
        if ds_name=='dm':
            var_list=list(set(groupby_colnames[ds_name])&set(vars_for_analysis_in_phase))
            #print(var_list)
            for pat_id in common_ids:
                bulk_df=ds[ds['USUBJID']==pat_id]
                bulk_df.reset_index(inplace=True)
                ds_dict[pat_id]={}
                ds_dict[pat_id]['bulk_data']=bulk_df
                ds_dict[pat_id]['variables']=var_list
                ds_dict[pat_id]['grouped_data']=bulk_df[var_list]
            f = open('../data/ttp_'+ds_name+'_dict',"wb")
            pickle.dump(ds_dict,f)
            f.close()

        elif isinstance(groupby_colnames[ds_name],str):
            #groupby_cat=name of column that contains the variables, df is grouped by the variables
            groupby_cat=groupby_colnames[ds_name]
            for pat_id in common_ids:
                bulk_df=ds[(ds['USUBJID']==pat_id)]

                ## Subset bulk dataframe of patient to variables that are considered in the corresponding phase
                # Add ds_name in front of the variables in order to match them with data format of variables in 'vars_for_analysis_in_phase'
                
                # For the lab variables: as the standardised test names were split into 2 columns previously 
                # (numerical and categorical tests) create a new column ("STD_TEST")bringing them together 
                if ds_name=='lb':
                    bulk_df.loc[:,'STD_NUM_TEST']=[ds_name+'_'+x for x in bulk_df['STD_NUM_TEST'].astype(str)]
                    bulk_df.loc[:,'STD_CAT_TEST']=[ds_name+'_'+x for x in bulk_df['STD_CAT_TEST'].astype(str)]
                    bulk_df['STD_TEST']=bulk_df['STD_NUM_TEST']
                    bulk_df.loc[bulk_df['STD_NUM_TEST'].isna(),'STD_TEST']=bulk_df.loc[bulk_df['STD_NUM_TEST'].isna(),'STD_CAT_TEST']
                    groupby_cat='STD_TEST'
                    
                if ds_name!='lb':
                    bulk_df.loc[:,groupby_cat]=[ds_name+'_'+x for x in bulk_df[groupby_cat].astype(str)]
                
                ## Consider only vars in phase that have been selected previously
                bulk_df=bulk_df[(bulk_df[groupby_cat].isin(vars_for_analysis_in_phase))]
                #print(bulk_df.shape)

                # Add bulk dataframe
                ds_dict[pat_id]={}
                #ds_dict[pat_id]['bulk_data']=bulk_df

                # Group bulk by variables and and add the variable names and the grouped dataframes to the dictionary
                gr=bulk_df.groupby(by=groupby_cat)
                ds_dict[pat_id]['variables']=list(gr.groups.keys())
                ds_dict[pat_id]['grouped_data']=gr
            f = open('../data/out_'+ds_name+'_dict',"wb")
            pickle.dump(ds_dict,f)
            f.close()

end = time.time()
print((end - start)/60)

mb
0.5752217928568523


In [7]:
del ds_dict

In [13]:
mb

Unnamed: 0          USUBJID  MBDY_estimated  STUDYID MBTESTCD  \
0                0  TB-1018/01-9002            -9.0  TB-1018      AFB   
1                1  TB-1018/01-9002             1.0  TB-1018      MTB   
2                2  TB-1018/01-9002             7.0  TB-1018      MTB   
3                3  TB-1018/01-9002            14.0  TB-1018      MTB   
4                4  TB-1018/01-9002            28.0  TB-1018      MTB   
...            ...              ...             ...      ...      ...   
173132       41151    TB-1022/53506           114.0  TB-1022      MTB   
173133       41152    TB-1022/53506           200.0  TB-1022      AFB   
173134       41153    TB-1022/53506           201.0  TB-1022      AFB   
173135       41154    TB-1022/53506           201.0  TB-1022      MTB   
173136       41155    TB-1022/53506           223.0  TB-1022      MTB   

                 MBTSTDTL                   MBMETHOD STD_RESULT  STD_MBTEST  \
0       Categorical Count            ACID FAST STAIN   negative    ZN-smear   
1          Culture Growth  MICROBIAL CULTURE, LIQUID   negative        MGIT   
2          Culture Growth  MICROBIAL CULTURE, LIQUID   negative        MGIT   
3          Culture Growth  MICROBIAL CULTURE, LIQUID   negative        MGIT   
4          Culture Growth  MICROBIAL CULTURE, LIQUID   positive        MGIT   
...                   ...                        ...        ...         ...   
173132     Culture Growth   MICROBIAL CULTURE, SOLID   negative  LJ-culture   
173133  Categorical Count                        NaN   positive    ZN-smear   
173134  Categorical Count                        NaN   positive    ZN-smear   
173135     Culture Growth   MICROBIAL CULTURE, SOLID   positive  LJ-culture   
173136     Culture Growth   MICROBIAL CULTURE, SOLID   positive  LJ-culture   

       SPDEVID SAMPLE_REFID                  MEDIATYP  STD_NUM_RESULT  \
0          NaN          NaN                       NaN             NaN   
1          NaN          NaN                       NaN             NaN   
2          NaN          NaN                       NaN             NaN   
3          NaN          NaN                       NaN             NaN   
4          NaN          NaN                       NaN             NaN   
...        ...          ...                       ...             ...   
173132     NaN          NaN  LOWENSTEIN JENSEN MEDIUM             NaN   
173133     NaN          NaN                       NaN             NaN   
173134     NaN          NaN                       NaN             NaN   
173135     NaN          NaN  LOWENSTEIN JENSEN MEDIUM             NaN   
173136     NaN          NaN  LOWENSTEIN JENSEN MEDIUM             NaN   

       STD_CAT_RESULT STD_NUM_UNITS  
0                 NaN           NaN  
1                 NaN           NaN  
2                 NaN           NaN  
3                 NaN           NaN  
4                 NaN           NaN  
...               ...           ...  
173132            NaN           NaN  
173133            NaN           NaN  
173134            NaN           NaN  
173135            NaN           NaN  
173136            NaN           NaN  

[173137 rows x 15 columns]